LightGBM, XGBoost, CatBoost는 모두 그래디언트 부스팅 프레임워크이며, **과적합(Overfitting) 방지**를 위해 조정해야 하는 핵심 파라미터들.

---

### 주요 과적합 방지 파라미터 비교

| 범주 | XGBoost | LightGBM | CatBoost | 핵심 역할 및 설명 |
| --- | --- | --- | --- | --- |
| **나무 복잡도 제어 (규제)** | `max_depth` | `max_depth` | `max_depth` | **나무의 최대 깊이.** 깊이가 깊을수록 모델 복잡도가 증가하며, 과적합 위험이 커집니다. 보통 3~10 사이의 값을 사용합니다. |
|  | `min_child_weight` | `min_child_samples` | `min_data_in_leaf` | **리프 노드에 필요한 최소 데이터 수(또는 가중치 합).** 이 값보다 적은 데이터를 가진 리프 노드는 생성되지 않습니다. 값이 클수록 나무의 성장을 억제하여 과적합을 방지합니다. |
|  | `gamma` (min_split_loss) | `min_gain_to_split` | `min_split_strength` (덜 일반적) | **노드를 분할하기 위해 필요한 최소 손실 감소 값.** 값이 클수록 분할에 더 엄격해져 나무의 복잡도를 낮춥니다. |
| **학습률 (Shrinkage)** | `learning_rate` (`eta`) | `learning_rate` | `learning_rate` | **각 부스팅 스텝의 기여도를 제어.** 작은 `learning_rate`를 사용하고 `n_estimators` (나무의 수)를 늘리는 것이 일반적입니다. 이를 통해 모델 학습 속도를 늦추고 일반화 성능을 높입니다. |
| **샘플링/피처 사용 규제** | `subsample` | `subsample` (`bagging_fraction`) | CatBoost는 내부적으로 샘플링을 수행 (`bootstrap_type`) | **각 나무를 학습할 때 사용할 데이터 샘플의 비율.** 1보다 작은 값을 사용하여 무작위 샘플링(Bagging) 효과를 주어 과적합을 방지합니다. (예: 0.8) |
|  | `colsample_bytree` | `colsample_bytree` (`feature_fraction`) | `colsample_bylevel` | **각 나무를 학습할 때 사용할 피처(특성)의 비율.** 1보다 작은 값을 사용하여 피처의 무작위 샘플링(Random Subspace) 효과를 주어 과적합을 방지합니다. (예: 0.8) |
| **L1/L2 규제** | `lambda` (`reg_lambda`, L2) `alpha` (`reg_alpha`, L1) | `lambda_l2` `lambda_l1` | `l2_leaf_reg` | **가중치에 대한 L1 및 L2 규제.** 계수(Coefficients) 크기를 제약하여 모델 복잡도를 낮추고 과적합을 방지합니다. |

---

### 모델별 추가적인 필수 파라미터

#### 1. LightGBM (Leaf-wise Growth)

LightGBM은 다른 모델과 달리 **리프 중심(Leaf-wise)** 성장 방식을 사용하므로, `max_depth` 외에 다음 파라미터들이 중요합니다.

* **`num_leaves`**: 리프 중심 방식의 **최대 리프 노드 수**. `max_depth`보다 이 파라미터가 나무의 복잡도를 더 직접적으로 제어합니다. 작은 값을 설정할수록 모델의 복잡도가 낮아집니다.

* **`min_data_in_leaf`**: 위에 설명된 대로 나무 성장을 제어합니다.

#### 2. CatBoost (Ordered Boosting & Categorical Features)

CatBoost는 내부적으로 Ordered Boosting과 카테고리형 피처 처리를 위한 특별한 방법을 사용하며, 과적합 방지를 위해 다음 파라미터가 중요합니다.

* **`one_hot_max_size`**: **원-핫 인코딩을 적용할 카테고리 값의 최대 개수.** 이 값을 너무 높게 설정하면 모델이 복잡해지고 과적합될 수 있습니다. (CatBoost는 기본적으로 특성 조합을 통해 카테고리형 피처를 처리합니다.)
* **`border_count` / `feature_border_type**`: 연속형 피처의 **분할 경계 수**를 제어합니다. 값이 클수록 더 많은 분할 옵션을 고려하게 되어 과적합 위험이 증가합니다.

---

###  권장 전략대부분의 경우, 과적합 방지를 위해 **가장 먼저 튜닝해야 할 핵심 파라미터**는 다음과 같습니다.

1. **`learning_rate`** (작게)
2. **`n_estimators`** (크게, Early Stopping과 함께 사용)
3. **`max_depth`** 또는 **`num_leaves`** (작게/적당히)
4. **`min_child_weight`** / **`min_data_in_leaf`** (크게)
5. **`subsample`** 및 **`colsample_bytree`** (1보다 작은 값, 예: 0.8)

이 파라미터들을 조정하는 하이퍼파라미터 튜닝(Grid Search등)을 수행하여 최적의 일반화 성능을 찾아야 한다.